In [1]:
import cv2
import numpy as np
from pathlib import Path
from tensorflow.keras import layers, models

In [2]:
root_fight = Path("/Users/mukuldixit/desktop/Projects/smartEye/data/raw/RWF-2000/train/Fight")
root_non = Path("/Users/mukuldixit/desktop/Projects/smartEye/data/raw/RWF-2000/train/NonFight")

In [3]:
def get_video_files(path):
    extensions = ['*.mp4', '*.avi', '*.MP4', '*.AVI']
    files = []
    for ext in extensions:
        files.extend(list(path.glob(ext)))
    return sorted(files)

fight_files = get_video_files(root_fight)[:200]
non_fight_files = get_video_files(root_non)[:200]

In [ ]:
def extract_frames(video_path, num_frames=80):
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []

    if total_frames < num_frames:
        cap.release()
        return None

    frame_indices = np.linspace(0, total_frames-1, num_frames).astype(int)

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            cap.release()
            return None

        frame = cv2.resize(frame, (128,128))
        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    return np.array(frames)

: 

In [ ]:
x = []
y = []

print("Processing Fight Videos...")

for video in fight_files:
    frames = extract_frames(video, 80)

    if frames is not None:
        x.append(frames)
        y.append(1)

print("Processing Non-Fight Videos...")

for video in non_fight_files:
    frames = extract_frames(video, 80)

    if frames is not None:
        x.append(frames)
        y.append(0)

x = np.array(x).astype("float32")
y = np.array(y)

print("Dataset Shape:")
print("X:", x.shape)
print("Y:", y.shape)

Processing Fight Videos...
Processing Non-Fight Videos...


In [ ]:
model = models.Sequential([

    layers.TimeDistributed(
        layers.Conv2D(32,(3,3),activation='relu'),
        input_shape=(80,128,128,3)
    ),
    layers.TimeDistributed(layers.MaxPooling2D(2,2)),

    layers.TimeDistributed(layers.Conv2D(64,(3,3),activation='relu')),
    layers.TimeDistributed(layers.MaxPooling2D(2,2)),

    layers.TimeDistributed(layers.Conv2D(128,(3,3),activation='relu')),
    layers.TimeDistributed(layers.MaxPooling2D(2,2)),

    layers.TimeDistributed(layers.Flatten()),

    layers.LSTM(64),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    x,
    y,
    validation_split=0.2,
    epochs=20,
    batch_size=4
)